<a href="https://colab.research.google.com/github/ThisalFernando/Fake-Real-News-Detection/blob/Random-Forest/RFA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
# ===============================
# 1. Import Libraries
# ===============================
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')

# ===============================
# 2. Load Dataset
# ===============================
df = pd.read_csv("improved_fake_news_dataset2.csv")

print("Dataset Shape:", df.shape)
print(df.head())

# ===============================
# 3. Check Columns
# ===============================
print(df.columns)

# ===============================
# 4. Clean Text Function
# ===============================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = text.split()
    text = [word for word in text if word not in stopwords.words('english')]
    return " ".join(text)

# ===============================
# 5. Apply Cleaning
# ===============================
df['clean_text'] = df['text'].apply(clean_text)

# ===============================
# 6. TF-IDF Vectorization
# ===============================
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_text']).toarray()
y = df['label']

# ===============================
# 7. Train-Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===============================
# 8. Random Forest Model
# ===============================
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# ===============================
# 9. Predictions
# ===============================
y_pred = rf_model.predict(X_test)

# ===============================
# 10. Evaluation
# ===============================
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ===============================
# 11. Custom Prediction
# ===============================
def predict_news(news):
    news_clean = clean_text(news)
    news_vector = vectorizer.transform([news_clean]).toarray()
    prediction = rf_model.predict(news_vector)

    if prediction[0] == 1:
        return "Real News"
    else:
        return "Fake News"

print(predict_news("Government announces new policy"))
print(predict_news("Aliens landed in Sri Lanka"))

In [ ]:
# 1. Handle missing values
df = df.dropna(subset=['text'])

# 2. Combine title and text
df['combined_text'] = df['title'].fillna('') + ' ' + df['text']
df['clean_text'] = df['combined_text'].apply(clean_text)

# 3. Check class balance
print(df['label'].value_counts())

# 4. Improve Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,        # More trees
    max_depth=20,            # Prevent overfitting
    min_samples_split=5,     # More conservative splits
    class_weight='balanced', # Handle imbalance
    random_state=42
)

# 5. Use cross-validation
from sklearn.model_selection import cross_val_score
scores = cross_val_score(rf_model, X_train, y_train, cv=5)
print(f"Cross-validation scores: {scores}")

# 6. Adjust classification threshold if needed
# Try adjusting predict probability threshold

In [ ]:
!pip install streamlit pyngrok

In [ ]:
# ===============================================
# 🔍 FAKE NEWS DETECTOR - INTERACTIVE GUI
# ===============================================

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
from datetime import datetime
import json
import os

nltk.download('stopwords')

# ===============================================
# 1. LOAD & TRAIN MODEL (Do this once)
# ===============================================
print("⏳ Loading dataset and training model...")

df = pd.read_csv("improved_fake_news_dataset2.csv")

# Handle missing values
df = df.dropna(subset=['text'])

# Combine title and text
df['combined_text'] = df['title'].fillna('') + ' ' + df['text']

# Clean text function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = text.split()
    text = [word for word in text if word not in stopwords.words('english')]
    return " ".join(text)

df['clean_text'] = df['combined_text'].apply(clean_text)

# Vectorize
vectorizer = TfidfVectorizer(max_features=2000)
X = vectorizer.fit_transform(df['clean_text']).toarray()
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Model accuracy
y_pred = rf_model.predict(X_test)
model_accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Model Trained! Accuracy: {model_accuracy:.2%}")

# ===============================================
# 2. PREDICTION FUNCTION
# ===============================================
def predict_news_with_probability(news_text):
    """Predict if news is fake or real with probability"""
    if not news_text.strip():
        return None, None, None

    news_clean = clean_text(news_text)
    news_vector = vectorizer.transform([news_clean]).toarray()

    # Get prediction and probability
    prediction = rf_model.predict(news_vector)[0]
    probability = rf_model.predict_proba(news_vector)[0]

    fake_prob = probability[0] * 100
    real_prob = probability[1] * 100
    confidence = max(fake_prob, real_prob)

    return prediction, confidence, (fake_prob, real_prob)

# ===============================================
# 3. RESULTS STORAGE
# ===============================================
results_list = []
results_file = "prediction_results.csv"

def save_result(news_text, prediction, confidence, fake_prob, real_prob):
    """Save prediction result to CSV"""
    result = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'news_article': news_text[:100] + "..." if len(news_text) > 100 else news_text,
        'prediction': 'REAL NEWS' if prediction == 1 else 'FAKE NEWS',
        'confidence': f"{confidence:.2f}%",
        'fake_probability': f"{fake_prob:.2f}%",
        'real_probability': f"{real_prob:.2f}%"
    }
    results_list.append(result)

    df_results = pd.DataFrame(results_list)
    df_results.to_csv(results_file, index=False)
    return df_results

# ===============================================
# 4. CREATE INTERACTIVE GUI
# ===============================================

# Title
title = widgets.HTML("<h2 style='text-align:center; color:#2E86AB;'>🔍 FAKE NEWS DETECTOR 🔍</h2>")
subtitle = widgets.HTML("<p style='text-align:center; color:#666;'>Enter a news article to check if it's fake or real</p>")

# Input text area
text_input = widgets.Textarea(
    value='',
    placeholder='Paste your news article here...',
    description='News Article:',
    rows=6,
    style={'description_width': '100px'}
)

# Buttons
predict_button = widgets.Button(
    description='🔎 PREDICT',
    button_style='info',
    tooltip='Click to predict',
    icon='magnifying-glass'
)

clear_button = widgets.Button(
    description='🗑️ CLEAR',
    button_style='warning',
    tooltip='Clear text',
    icon='trash'
)

history_button = widgets.Button(
    description='📊 VIEW HISTORY',
    button_style='success',
    tooltip='View all predictions',
    icon='history'
)

# Output area
output_area = widgets.Output()

# ===============================================
# 5. BUTTON CLICK HANDLERS
# ===============================================

def on_predict_click(b):
    """Handle predict button click"""
    with output_area:
        clear_output(wait=True)

        news_text = text_input.value.strip()

        if not news_text:
            print("⚠️ Please enter a news article!")
            return

        print("⏳ Analyzing article...")
        prediction, confidence, (fake_prob, real_prob) = predict_news_with_probability(news_text)

        if prediction is None:
            print("⚠️ Please enter valid text!")
            return

        # Display results
        print("\n" + "="*60)
        print("📊 PREDICTION RESULTS")
        print("="*60)

        if prediction == 1:
            result_text = "✅ REAL NEWS"
            color = "green"
        else:
            result_text = "❌ FAKE NEWS"
            color = "red"

        print(f"\n{result_text}")
        print(f"Confidence: {confidence:.2f}%")
        print(f"\nProbability Breakdown:")
        print(f"  Fake News: {fake_prob:.2f}%")
        print(f"  Real News: {real_prob:.2f}%")

        # Create probability chart
        fig, ax = plt.subplots(figsize=(10, 4))
        categories = ['Fake News', 'Real News']
        probabilities = [fake_prob, real_prob]
        colors = ['#E63946', '#06A77D']

        bars = ax.barh(categories, probabilities, color=colors)
        ax.set_xlim(0, 100)
        ax.set_xlabel('Probability (%)', fontsize=12, fontweight='bold')
        ax.set_title('Fake News Detection - Probability Distribution', fontsize=14, fontweight='bold')

        # Add percentage labels on bars
        for i, (bar, prob) in enumerate(zip(bars, probabilities)):
            ax.text(prob + 2, i, f'{prob:.2f}%', va='center', fontweight='bold')

        plt.tight_layout()
        plt.show()

        # Save result
        save_result(news_text, prediction, confidence, fake_prob, real_prob)
        print("\n💾 Result saved to CSV!")
        print("="*60)

def on_clear_click(b):
    """Handle clear button click"""
    text_input.value = ''
    with output_area:
        clear_output(wait=True)
        print("✨ Cleared!")

def on_history_click(b):
    """Handle history button click"""
    with output_area:
        clear_output(wait=True)

        if not results_list:
            print("📭 No predictions yet!")
            return

        df_results = pd.read_csv(results_file)
        print("\n" + "="*100)
        print("📋 PREDICTION HISTORY")
        print("="*100)
        print(df_results.to_string(index=False))
        print("="*100)
        print(f"\n📊 Total Predictions: {len(df_results)}")

# Attach click handlers
predict_button.on_click(on_predict_click)
clear_button.on_click(on_clear_click)
history_button.on_click(on_history_click)

# ===============================================
# 6. DISPLAY GUI
# ===============================================

# Button container
button_box = widgets.HBox([predict_button, clear_button, history_button])

# Main container
gui = widgets.VBox([
    title,
    subtitle,
    widgets.HTML("<hr>"),
    text_input,
    button_box,
    widgets.HTML("<hr>"),
    output_area
])

display(gui)

print("\n✅ GUI Ready! Enter a news article and click PREDICT")

In [ ]:
def predict_news_with_probability(news_text, threshold=0.75):
    """Predict with custom threshold"""
    if not news_text.strip():
        return None, None, None

    news_clean = clean_text(news_text)
    news_vector = vectorizer.transform([news_clean]).toarray()

    probability = rf_model.predict_proba(news_vector)[0]
    fake_prob = probability[0] * 100
    real_prob = probability[1] * 100

    # Use custom threshold instead of 0.5
    prediction = 1 if real_prob >= threshold else 0
    confidence = max(fake_prob, real_prob)

    return prediction, confidence, (fake_prob, real_prob)

# Change threshold to 75%
# predict_news_with_probability(text, threshold=0.75)

In [ ]:
import pickle

# Save model
pickle.dump(rf_model, open('fake_news_model.pkl', 'wb'))
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))
print("✅ Model saved!")

In [ ]:
test_cases = [
    "Aliens have attack sri lanka",  # Should be FAKE now
    "Government announces new policy",  # Should be REAL
    "Breaking: Scientists discover cure for cancer",  # Should be REAL
    "The earth is flat and NASA is lying",  # Should be FAKE
    "President visits local school",  # Should be REAL
    "Lizard people control the world",  # Should be FAKE
]

for news in test_cases:
    pred, conf, (fake_p, real_p) = predict_news_with_probability(news, threshold=0.75)
    result = "✅ REAL" if pred == 1 else "❌ FAKE"
    print(f"'{news[:40]}...' → {result} ({conf:.2f}%)")

In [ ]:
# Debug: Print raw probabilities WITHOUT threshold
test_news = "Aliens have attack sri lanka"
news_clean = clean_text(test_news)
news_vector = vectorizer.transform([news_clean]).toarray()

# Raw prediction
raw_pred = rf_model.predict(news_vector)
raw_proba = rf_model.predict_proba(news_vector)

print(f"Raw Prediction (0 or 1): {raw_pred[0]}")
print(f"Raw Probabilities: {raw_proba[0]}")
print(f"  Fake (0): {raw_proba[0][0] * 100:.2f}%")
print(f"  Real (1): {raw_proba[0][1] * 100:.2f}%")

In [ ]:
# Test with HIGH threshold (90%)
threshold = 0.90
test_news = "Aliens have attack sri lanka"
news_clean = clean_text(test_news)
news_vector = vectorizer.transform([news_clean]).toarray()
raw_proba = rf_model.predict_proba(news_vector)[0]

prediction = 1 if raw_proba[1] >= threshold else 0
result = "✅ REAL" if prediction == 1 else "❌ FAKE"

print(f"Real%: {raw_proba[1]*100:.2f}%")
print(f"Threshold: {threshold*100:.0f}%")
print(f"Result: {result}")

In [ ]:
print("=== TEST 3: Data Balance ===")
print(f"Fake in train: {(y_train == 0).sum()}")
print(f"Real in train: {(y_train == 1).sum()}")
print(f"\nFake in test: {(y_test == 0).sum()}")
print(f"Real in test: {(y_test == 1).sum()}")

In [3]:
from google.colab import files
print("📁 Upload your CSV file:")
uploaded = files.upload()
print("✅ File uploaded!")

📁 Upload your CSV file:


Saving improved_fake_news_dataset2.csv to improved_fake_news_dataset2.csv
✅ File uploaded!


In [4]:
# ===============================================
# 🔍 FAKE NEWS DETECTOR - INTERACTIVE GUI v2.0
# ===============================================

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
from datetime import datetime
import json
import os

nltk.download('stopwords')

# ===============================================
# 1. LOAD & TRAIN MODEL
# ===============================================
print("⏳ Loading dataset and training model...")

df = pd.read_csv("improved_fake_news_dataset2.csv")

# Handle missing values
df = df.dropna(subset=['text'])

# Combine title and text
df['combined_text'] = df['title'].fillna('') + ' ' + df['text']

# Clean text function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = text.split()
    text = [word for word in text if word not in stopwords.words('english')]
    return " ".join(text)

df['clean_text'] = df['combined_text'].apply(clean_text)

# Vectorize
vectorizer = TfidfVectorizer(max_features=2000)
X = vectorizer.fit_transform(df['clean_text']).toarray()
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Model accuracy
y_pred = rf_model.predict(X_test)
model_accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Model Trained! Accuracy: {model_accuracy:.2%}")

# ===============================================
# 2. PREDICTION FUNCTION WITH 0.90 THRESHOLD
# ===============================================
def predict_news_with_probability(news_text, threshold=0.90):
    """Predict if news is fake or real with probability"""
    if not news_text.strip():
        return None, None, None

    news_clean = clean_text(news_text)
    news_vector = vectorizer.transform([news_clean]).toarray()

    # Get prediction and probability
    probability = rf_model.predict_proba(news_vector)[0]

    fake_prob = probability[0] * 100
    real_prob = probability[1] * 100

    # Use 0.90 threshold instead of default 0.50
    prediction = 1 if real_prob >= (threshold * 100) else 0
    confidence = max(fake_prob, real_prob)

    return prediction, confidence, (fake_prob, real_prob)

# ===============================================
# 3. RESULTS STORAGE
# ===============================================
results_list = []
results_file = "prediction_results.csv"

def save_result(news_text, prediction, confidence, fake_prob, real_prob):
    """Save prediction result to CSV"""
    result = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'news_article': news_text[:100] + "..." if len(news_text) > 100 else news_text,
        'prediction': 'REAL NEWS' if prediction == 1 else 'FAKE NEWS',
        'confidence': f"{confidence:.2f}%",
        'fake_probability': f"{fake_prob:.2f}%",
        'real_probability': f"{real_prob:.2f}%"
    }
    results_list.append(result)

    df_results = pd.DataFrame(results_list)
    df_results.to_csv(results_file, index=False)
    return df_results

# ===============================================
# 4. CREATE INTERACTIVE GUI
# ===============================================

# Title
title = widgets.HTML("<h2 style='text-align:center; color:#2E86AB;'>🔍 FAKE NEWS DETECTOR 🔍</h2>")
subtitle = widgets.HTML("<p style='text-align:center; color:#666;'>Enter a news article to check if it's fake or real</p>")
model_info = widgets.HTML(f"<p style='text-align:center; color:#888;'>Model Accuracy: {model_accuracy:.2%} | Threshold: 90%</p>")

# Input text area
text_input = widgets.Textarea(
    value='',
    placeholder='Paste your news article here...',
    description='News Article:',
    rows=6,
    style={'description_width': '100px'}
)

# Buttons
predict_button = widgets.Button(
    description='🔎 PREDICT',
    button_style='info',
    tooltip='Click to predict',
    icon='magnifying-glass'
)

clear_button = widgets.Button(
    description='🗑️ CLEAR',
    button_style='warning',
    tooltip='Clear text',
    icon='trash'
)

history_button = widgets.Button(
    description='📊 VIEW HISTORY',
    button_style='success',
    tooltip='View all predictions',
    icon='history'
)

# Output area
output_area = widgets.Output()

# ===============================================
# 5. BUTTON CLICK HANDLERS
# ===============================================

def on_predict_click(b):
    """Handle predict button click"""
    with output_area:
        clear_output(wait=True)

        news_text = text_input.value.strip()

        if not news_text:
            print("⚠️ Please enter a news article!")
            return

        print("⏳ Analyzing article...")
        prediction, confidence, (fake_prob, real_prob) = predict_news_with_probability(news_text, threshold=0.90)

        if prediction is None:
            print("⚠️ Please enter valid text!")
            return

        # Display results
        print("\n" + "="*60)
        print("📊 PREDICTION RESULTS")
        print("="*60)

        if prediction == 1:
            result_text = "✅ REAL NEWS"
            color = "green"
        else:
            result_text = "❌ FAKE NEWS"
            color = "red"

        print(f"\n{result_text}")
        print(f"Confidence: {confidence:.2f}%")
        print(f"\nProbability Breakdown:")
        print(f"  Fake News: {fake_prob:.2f}%")
        print(f"  Real News: {real_prob:.2f}%")
        print(f"\nThreshold Used: 90%")

        # Create probability chart
        fig, ax = plt.subplots(figsize=(10, 4))
        categories = ['Fake News', 'Real News']
        probabilities = [fake_prob, real_prob]
        colors = ['#E63946', '#06A77D']

        bars = ax.barh(categories, probabilities, color=colors)
        ax.set_xlim(0, 100)
        ax.set_xlabel('Probability (%)', fontsize=12, fontweight='bold')
        ax.set_title('Fake News Detection - Probability Distribution', fontsize=14, fontweight='bold')

        # Add percentage labels on bars
        for i, (bar, prob) in enumerate(zip(bars, probabilities)):
            ax.text(prob + 2, i, f'{prob:.2f}%', va='center', fontweight='bold')

        plt.tight_layout()
        plt.show()

        # Save result
        save_result(news_text, prediction, confidence, fake_prob, real_prob)
        print("\n💾 Result saved to CSV!")
        print("="*60)

def on_clear_click(b):
    """Handle clear button click"""
    text_input.value = ''
    with output_area:
        clear_output(wait=True)
        print("✨ Cleared!")

def on_history_click(b):
    """Handle history button click"""
    with output_area:
        clear_output(wait=True)

        if not results_list:
            print("📭 No predictions yet!")
            return

        df_results = pd.read_csv(results_file)
        print("\n" + "="*120)
        print("📋 PREDICTION HISTORY")
        print("="*120)
        print(df_results.to_string(index=False))
        print("="*120)
        print(f"\n📊 Total Predictions: {len(df_results)}")

        # Statistics
        real_count = (df_results['prediction'] == 'REAL NEWS').sum()
        fake_count = (df_results['prediction'] == 'FAKE NEWS').sum()
        print(f"✅ Real News: {real_count}")
        print(f"❌ Fake News: {fake_count}")

# Attach click handlers
predict_button.on_click(on_predict_click)
clear_button.on_click(on_clear_click)
history_button.on_click(on_history_click)

# ===============================================
# 6. DISPLAY GUI
# ===============================================

# Button container
button_box = widgets.HBox([predict_button, clear_button, history_button])

# Main container
gui = widgets.VBox([
    title,
    subtitle,
    model_info,
    widgets.HTML("<hr>"),
    text_input,
    button_box,
    widgets.HTML("<hr>"),
    output_area
])

display(gui)

print("\n✅ GUI Ready! Enter a news article and click PREDICT")
print("📌 Threshold set to 90% for stricter predictions")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


⏳ Loading dataset and training model...
✅ Model Trained! Accuracy: 80.26%



✅ GUI Ready! Enter a news article and click PREDICT
📌 Threshold set to 90% for stricter predictions


In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# ===============================================
# GET ALL METRICS FOR RANDOM FOREST
# ===============================================

# Make predictions on test set
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]  # Probability for class 1 (REAL)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Display results
print("=" * 70)
print("🎯 RANDOM FOREST MODEL - DETAILED METRICS")
print("=" * 70)
print(f"Accuracy:    {accuracy:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-Score:    {f1:.4f}")
print(f"ROC-AUC:     {roc_auc:.4f}")
print("=" * 70)

# Detailed classification report
print("\n📋 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=['Fake News', 'Real News']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\n📊 CONFUSION MATRIX:")
print(f"True Negatives:  {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives:  {cm[1][1]}")

🎯 RANDOM FOREST MODEL - DETAILED METRICS
Accuracy:    0.8026
Precision:   0.8206
Recall:      0.7771
F1-Score:    0.7983
ROC-AUC:     0.9025

📋 CLASSIFICATION REPORT:
              precision    recall  f1-score   support

   Fake News       0.79      0.83      0.81     12333
   Real News       0.82      0.78      0.80     12458

    accuracy                           0.80     24791
   macro avg       0.80      0.80      0.80     24791
weighted avg       0.80      0.80      0.80     24791


📊 CONFUSION MATRIX:
True Negatives:  10217
False Positives: 2116
False Negatives: 2777
True Positives:  9681


In [6]:
from sklearn.metrics import log_loss

# Predict probabilities on validation/test set
y_val_proba = rf_model.predict_proba(X_test)

# Calculate log loss
val_loss = log_loss(y_test, y_val_proba)

print(f"Validation Loss (Log Loss): {val_loss:.4f}")

Validation Loss (Log Loss): 0.4652
